# Day 034 — Exercise 4: dispatch

**What you'll build:** `dispatch(ns, handlers) -> str` — look up `ns.command` in a `handlers` dict and call the matching handler, returning its result. Raise `KeyError` if the command has no handler.

**Why it matters:** The `handlers` dict pattern is extensible (add a command = add one dict entry), testable (inject mock handlers), and explicit (KeyError immediately flags a missing handler).

## Provided: make_subcommand_parser (used in checks)

In [ ]:
from argparse import ArgumentParser

def make_parser() -> ArgumentParser:
    parser = ArgumentParser(
        prog='ai-tool',
        description='AI command-line tool powered by local LLM',
    )
    parser.add_argument(
        '--prompt', '-p', type=str, required=True,
        help='Prompt to send to the model',
    )
    parser.add_argument(
        '--model', '-m', type=str, default='llama3.2',
        help='Ollama model name (default: llama3.2)',
    )
    parser.add_argument(
        '--verbose', '-v', action='store_true',
        help='Print extra diagnostic output',
    )
    return parser


def make_subcommand_parser() -> ArgumentParser:
    parser = ArgumentParser(prog='ai-tool', description='AI CLI')
    subs   = parser.add_subparsers(dest='command', required=True,
                                   title='commands')

    chat = subs.add_parser('chat', help='Send a prompt to the AI')
    chat.add_argument('--prompt', '-p', required=True, help='The prompt')
    chat.add_argument('--model',  '-m', default='llama3.2')

    summarize = subs.add_parser('summarize', help='Summarize text')
    summarize.add_argument('--text',  '-t', required=True,
                           help='Text to summarize')
    summarize.add_argument('--model', '-m', default='llama3.2')

    return parser

## Your Implementation

In [ ]:
def dispatch(ns, handlers: dict) -> str:
    """
    Route a parsed Namespace to the correct handler function.

    Args:
        ns:       Parsed argparse Namespace with a .command attribute.
        handlers: dict mapping command name (str) to handler function.

    Returns:
        The string returned by handlers[ns.command](ns).

    Raises:
        KeyError: if ns.command is not in handlers.
    """
    # TODO: cmd = ns.command
    # TODO: if cmd not in handlers:
    #     raise KeyError(f'No handler registered for command: {cmd!r}')
    # TODO: return handlers[cmd](ns)
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Stub handlers for testing — no LLM calls
    def _chat_handler(ns):
        return f'chat:{ns.prompt}'

    def _summarize_handler(ns):
        return f'summarize:{ns.text}'

    handlers = {
        'chat':      _chat_handler,
        'summarize': _summarize_handler,
    }
    parser = make_subcommand_parser()

    # Check 1: defined
    try:
        assert 'dispatch' in globals()
        passed += 1; print('\u2705 Check 1: dispatch defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}')
        return

    # Check 2: routes 'chat' correctly
    try:
        ns = parser.parse_args(['chat', '--prompt', 'hello'])
        result = dispatch(ns, handlers)
        assert result == 'chat:hello', \
            f"expected 'chat:hello', got {result!r}"
        passed += 1; print('\u2705 Check 2: dispatch routes chat → _chat_handler')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: routes 'summarize' correctly
    try:
        ns = parser.parse_args(['summarize', '--text', 'some text'])
        result = dispatch(ns, handlers)
        assert result == 'summarize:some text', \
            f"expected 'summarize:some text', got {result!r}"
        passed += 1; print('\u2705 Check 3: dispatch routes summarize → _summarize_handler')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: raises KeyError for unknown command
    try:
        import types
        fake_ns = types.SimpleNamespace(command='classify', text='x')
        raised = False
        try:
            dispatch(fake_ns, handlers)
        except KeyError:
            raised = True
        assert raised, 'unknown command should raise KeyError'
        passed += 1; print('\u2705 Check 4: unknown command raises KeyError')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: handler receives correct namespace attributes
    try:
        received = {}
        def _inspecting_handler(ns):
            received['command'] = ns.command
            received['prompt']  = ns.prompt
            received['model']   = ns.model
            return 'ok'
        h2 = {'chat': _inspecting_handler}
        ns = parser.parse_args(['chat', '--prompt', 'test', '--model', 'custom'])
        dispatch(ns, h2)
        assert received['command'] == 'chat',   f'command: {received.get("command")!r}'
        assert received['prompt']  == 'test',   f'prompt: {received.get("prompt")!r}'
        assert received['model']   == 'custom', f'model: {received.get("model")!r}'
        passed += 1; print('\u2705 Check 5: handler receives full namespace with correct attrs')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def dispatch(ns, handlers: dict) -> str:
    cmd = ns.command
    if cmd not in handlers:
        raise KeyError(f"No handler registered for command: {cmd!r}")
    return handlers[cmd](ns)
```

</details>